In [1]:
import pandas as pd


In [3]:
import pandas as pd

df = pd.read_csv("data/SP500.csv")


In [6]:
df.tail()


,Date,Close,High,Low,Open,Volume
24527,8/25/2025,6439.319824,6466.890137,6438.060059,6457.669922,4059070000
24528,8/26/2025,6465.939941,6468.370117,6429.209961,6435.490234,4867680000
24529,8/27/2025,6481.399902,6487.060059,6457.839844,6462.259766,4143680000
24530,8/28/2025,6501.859863,6508.229980,6466.959961,6483.839844,4283760000
24531,8/29/2025,6460.259766,6491.759766,6444.569824,6489.279785,4234840000


In [9]:
df.shape


(24532, 6)

In [8]:
df.dtypes


Date          str
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object

In [11]:
df.columns


Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='str')

In [12]:
df.index


RangeIndex(start=0, stop=24532, step=1)

In [15]:
df['Date'] = pd.to_datetime(df['Date'])


In [16]:
df.dtypes


Date      datetime64[us]
Close            float64
High             float64
Low              float64
Open             float64
Volume             int64
dtype: object

In [17]:
print(df['Date'])


0       1927-12-30
1       1928-01-03
2       1928-01-04
3       1928-01-05
4       1928-01-06
           ...    
24527   2025-08-25
24528   2025-08-26
24529   2025-08-27
24530   2025-08-28
24531   2025-08-29
Name: Date, Length: 24532, dtype: datetime64[us]


In [18]:
df = df[df['Date'] >= "2010-01-01"]


In [19]:
df["Date"].is_monotonic_increasing


True

In [20]:
feature_cols = ["Open", "High", "Low", "Close", "Volume"]
target_col = "Close"

print("Features:", feature_cols)
print("Target:", target_col)


Features: ['Open', 'High', 'Low', 'Close', 'Volume']
Target: Close


In [21]:
n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()


In [22]:
print("Total:", len(df))
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print(
    "Check:",
    len(train_df) + len(val_df) + len(test_df)
)


Total: 3939
Train: 2757
Validation: 591
Test: 591
Check: 3939


In [23]:
print(
    "Train:",
    train_df["Date"].min(),
    "->",
    train_df["Date"].max()
)

print(
    "Validation:",
    val_df["Date"].min(),
    "->",
    val_df["Date"].max()
)

print(
    "Test:",
    test_df["Date"].min(),
    "->",
    test_df["Date"].max()
)


Train: 2010-01-04 00:00:00 -> 2020-12-14 00:00:00
Validation: 2020-12-15 00:00:00 -> 2023-04-21 00:00:00
Test: 2023-04-24 00:00:00 -> 2025-08-29 00:00:00


In [24]:
assert len(train_df) + len(val_df) + len(test_df) == len(df)

assert train_df["Date"].max() < val_df["Date"].min()
assert val_df["Date"].max() < test_df["Date"].min()


In [25]:
import numpy as np
import torch

from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

feature_scaler.fit(train_df[feature_cols])
target_scaler.fit(train_df[[target_col]])

X_train_scaled = feature_scaler.transform(train_df[feature_cols])
X_val_scaled = feature_scaler.transform(val_df[feature_cols])
X_test_scaled = feature_scaler.transform(test_df[feature_cols])

y_train_scaled = target_scaler.transform(train_df[[target_col]])
y_val_scaled = target_scaler.transform(val_df[[target_col]])
y_test_scaled = target_scaler.transform(test_df[[target_col]])

print("X_train:", X_train_scaled.shape)
print("y_train:", y_train_scaled.shape)

print("X_val:", X_val_scaled.shape)
print("y_val:", y_val_scaled.shape)

print("X_test:", X_test_scaled.shape)
print("y_test:", y_test_scaled.shape)


X_train: (2757, 5)
y_train: (2757, 1)
X_val: (591, 5)
y_val: (591, 1)
X_test: (591, 5)
y_test: (591, 1)


In [ ]:
WINDOW_SIZE = 20

def create_sequences(X, y, window_size):
    X_seq = []
    y_seq = []

    for i in range(len(X) - window_size):

        X_window = X[i:i + window_size]

        y_target = y[i + window_size]

        X_seq.append(X_window)
        y_seq.append(y_target)

    return np.array(X_seq), np.array(y_seq)
